# Phase 0 — EDA: CFPB Complaint Classification

Class balance, narrative length, and label-noise discussion on the processed train split.
Run `python -m src.data.acquire` then `python -m src.data.preprocess` first to produce `data/processed/train.csv`.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
from src.data.preprocess import LABEL_COL, NARRATIVE_COL

train = pd.read_csv('../data/processed/train.csv')
len(train)

In [ ]:
counts = train[LABEL_COL].value_counts()
counts

In [ ]:
print(f"Imbalance ratio (max/min class): {counts.max()/counts.min():.1f}x")
counts.sort_values().plot(kind='barh', figsize=(8,5), color='#2b5876')
plt.title('Class balance — train split')
plt.xlabel('examples')
plt.tight_layout()
plt.show()

The 8-category label set is a deliberate collapse of CFPB's 20+ raw, multi-year `Product` taxonomy variants (see `src/data/preprocess.py::LABEL_MAP` for the full rationale: collapse only on historical renames/near-duplicates, never across substantively different products). Credit reporting complaints dominate — matching the real CFPB dataset's skew — which is why Phase 2 fine-tuning uses class-weighted loss rather than plain cross-entropy.

In [ ]:
train['wordcount'] = train[NARRATIVE_COL].str.split().str.len()
train['wordcount'].describe()

In [ ]:
train['wordcount'].clip(upper=200).hist(bins=40, figsize=(8,5), color='#c17817')
plt.title('Narrative length distribution')
plt.xlabel('words per complaint (clipped at 200)')
plt.tight_layout()
plt.show()

## Label noise

Complaints pass through `collapse_labels()`, which maps the raw multi-year `Product` field onto 8 canonical categories. A portion of raw `Product` values are genuinely inconsistent with the complaint content — CFPB's own taxonomy has been applied inconsistently across years (see project scope Section 4.1). This is *not* scrubbed before training. The class-weighted loss in Phase 2 and the macro-F1 promotion gate in Phase 3 exist specifically to be robust to this, rather than assuming a clean label set that real production data never has.